In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
import numpy as np
import seaborn as sns

In [3]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [4]:
# 소수점 아래 10자리까지 출력
pd.options.display.float_format = '{:.10f}'.format

In [5]:
ns = 1e-9

In [12]:
base_dir=f'{username}/mgvm/results/reports'
date = "20260511-145025-mesh-6x6/data-allocation-logs"
data_alloc_dir = f"{base_dir}/{date}"
print(f"{data_alloc_dir}")

/home/ehdgus7279/mgvm/results/reports/20260511-145025-mesh-6x6/data-allocation-logs


In [39]:
benchmark = 'gpt3_gemm'

xDim=6
yDim=6

data_alloc_file = f"{data_alloc_dir}/{benchmark}_mesh_{xDim}x{yDim}_data_allocation_log.csv"

data_alloc_df = pd.read_csv(data_alloc_file, skipinitialspace=True)
print(f"{data_alloc_df.duplicated().sum()} duplicated rows found.")

data_alloc_df = data_alloc_df.drop_duplicates()
data_alloc_df

0 duplicated rows found.


,DataID,ChipletID,StartAddr,EndAddr
0,2,28,4586500096,4586504192
1,2,28,4586504192,4586508288
2,2,28,4586508288,4586512384
3,2,28,4586512384,4586516480
4,2,28,4586516480,4586520576
...,...,...,...,...
593659,1,35,4563386368,4563390464
593660,1,35,4563390464,4563394560
593661,1,35,4563394560,4563398656
593662,1,35,4563398656,4563402752


In [40]:
trace_base_dir=f'{username}/mgvm/results/memory-traces'
sub_dir = "20260506-173915-mesh-6x6"
trace_dir = f"{trace_base_dir}/{sub_dir}"
print(f"{trace_dir}")

/home/ehdgus7279/mgvm/results/memory-traces/20260506-173915-mesh-6x6


In [41]:
xDim=6
yDim=6

trace_file = f"{trace_dir}/{benchmark}_mesh_{xDim}x{yDim}_memory_trace.csv"

df = pd.read_csv(trace_file, skipinitialspace=True)
print(f"{df.duplicated().sum()} duplicated rows found.")

df = df.drop_duplicates()
df

0 duplicated rows found.


,time,addr,pc,owner,rw,rdma,vector,type,chiplet
0,0.0987477880,4598800384,2431648000,0,0,0,0,2,0
1,0.0987477880,4598800384,2431648000,0,0,0,0,0,0
2,0.0987477900,4598800384,2431648000,0,0,0,0,0,0
3,0.0987477920,4598800384,2431648000,0,0,0,0,0,0
4,0.0987477940,4598800384,2431648000,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...
103622239,0.0995493670,4592292928,2431648228,8,1,1,1,4,12
103622240,0.0995493670,4568791104,2431648228,10,1,1,1,4,15
103622241,0.0995493670,2148497088,2431648268,15,0,1,1,1,33
103622242,0.0995493670,4568343744,2431648228,3,1,1,1,4,23


In [42]:
rdma_df = df[df['rdma']==True]
rdma_df = rdma_df.sort_values(by='addr')

In [43]:
data_alloc_df = data_alloc_df.sort_values(by='StartAddr')

mapped_df = pd.merge_asof(
    rdma_df,
    data_alloc_df,
    left_on='addr',
    right_on='StartAddr',
    direction='backward'
)

invalid_mask = (mapped_df['addr'] >= mapped_df['EndAddr']) | mapped_df['StartAddr'].isna()
mapped_df.loc[invalid_mask, ['DataID', 'ChipletID']] = None

mapped_df = mapped_df.sort_values('time')

mapped_df

,time,addr,pc,owner,rw,rdma,vector,type,chiplet,DataID,ChipletID,StartAddr,EndAddr
100634874,0.0987478410,4598800384,2431648000,0,0,1,0,2,1,NaN,NaN,4598575104,4598579200
100634758,0.0987478410,4598800384,2431648000,0,0,1,0,0,1,NaN,NaN,4598575104,4598579200
100634895,0.0987478430,4598800384,2431648000,0,0,1,0,0,1,NaN,NaN,4598575104,4598579200
100634754,0.0987478450,4598800384,2431648000,0,0,1,0,0,1,NaN,NaN,4598575104,4598579200
100634752,0.0987478470,4598800384,2431648000,0,0,1,0,0,1,NaN,NaN,4598575104,4598579200
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97410454,0.0995493670,4594625984,2431648228,8,1,1,1,4,30,2.0000000000,8.0000000000,4594622464,4594626560
45518640,0.0995493670,4568585344,2431648228,7,1,1,1,4,28,2.0000000000,7.0000000000,4568584192,4568588288
22798498,0.0995493670,2150446592,2431648268,9,0,1,1,1,6,0.0000000000,9.0000000000,2150445056,2150449152
35992217,0.0995493670,2231081472,2431648284,15,0,1,1,2,4,1.0000000000,15.0000000000,2231078912,2231083008


In [44]:
mapped_df.groupby('DataID')['rw'].value_counts()

DataID        rw
0.0000000000  0     27035312
1.0000000000  0      9050486
2.0000000000  1     64369795
              0       179149
Name: count, dtype: int64

In [45]:
shared_chiplets = mapped_df.groupby('addr').agg({
    'addr': 'count',    # 해당 주소로의 총 접근 횟수
    'chiplet': 'nunique', # 공유된 칩렛 수
    'DataID': 'first' # DataID도 함께 유지
}).rename(columns={
    'addr': 'access_count', 
    'chiplet': 'unique_chiplets'
}).reset_index()

In [46]:
shared_chiplets.groupby(['DataID'])['unique_chiplets'].value_counts()

DataID        unique_chiplets
0.0000000000  35                    1456
              1                       55
              7                       34
              30                      23
              14                      21
              13                      19
              6                       15
              25                      15
              33                      15
              21                      14
              29                      13
              20                      12
              31                      10
              2                        9
              12                       9
              17                       9
              26                       9
              4                        8
              5                        7
              18                       7
              11                       6
              19                       6
              24                       6
              10           

In [47]:
shared_chiplets.groupby(['DataID', 'unique_chiplets'])['access_count'].sum()

DataID        unique_chiplets
0.0000000000  1                       383
              2                        32
              4                        56
              5                       233
              6                      3002
              7                     13477
              9                       548
              10                      802
              11                      652
              12                     3787
              13                    15496
              14                    34362
              17                     3621
              18                     3019
              19                    23702
              20                    56920
              21                    67881
              24                     1039
              25                    10650
              26                     6633
              27                      371
              29                   109664
              30                   197051
    

In [ ]:
mapped_df['rw'].value_counts()

rw
1    64369795
0    36265297
Name: count, dtype: int64

: 

In [41]:
mapped_df['DataID'].value_counts()

DataID
1.0000000000    27529216
0.0000000000     8801303
2.0000000000      107520
Name: count, dtype: int64

In [54]:
mapped_df['ChipletID'].value_counts()

ChipletID
0.0000000000     18381103
15.0000000000      467634
Name: count, dtype: int64